In [1]:
# ============================================================
# 09_bivariate_varima_rolling_eval.ipynb

# Purpose:
#   Bivariate ARI-ILI VARIMA-like benchmark by country using VARMAX, evaluated with explicit rolling-origin protocol on 2023.

# Methodology:
#   - Train: 2014-2022
#   - Test: 2023
#   - Countries: common valid ARI/ILI countries
#   - Horizons: h = 1,2,3,4
#   - Model: one bivariate model per country
#   - Variables: ARI and ILI jointly
#   - Metrics: MAE, RMSE, MASE
#   - Evaluation calendar is explicit and auditable.
# ============================================================

import sys
!"{sys.executable}" -m pip install -U statsmodels

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import get_project_paths
from src.io_data import read_ari_ili
from src.gaps import gap_summary
from src.impute import impute_series_weekly
from src.calendars import extract_raw_weekly_series
from src.metrics import mae, rmse, mase_scale_seasonal, mase_from_mae

from statsmodels.tsa.statespace.varmax import VARMAX

paths = get_project_paths(PROJECT_ROOT)
paths.results.mkdir(parents=True, exist_ok=True)

ari_df, ili_df = read_ari_ili(paths.data)

ari_df["truth_date"] = pd.to_datetime(ari_df["truth_date"])
ili_df["truth_date"] = pd.to_datetime(ili_df["truth_date"])

ari_df = ari_df.sort_values(["location", "truth_date"]).reset_index(drop=True)
ili_df = ili_df.sort_values(["location", "truth_date"]).reset_index(drop=True)

print("ARI shape:", ari_df.shape)
print("ILI shape:", ili_df.shape)


# ============================================================
# 1. Configuration and country selection
# ============================================================

MIN_OBS = 300
MAX_GAP = 8
M_SEASON = 52
HORIZONS = (1, 2, 3, 4)
MAX_H = max(HORIZONS)

gap_ari = gap_summary(ari_df, disease="ARI", calendar_mode="data_range")
gap_ili = gap_summary(ili_df, disease="ILI", calendar_mode="data_range")

ok_ari = gap_ari[
    (gap_ari["n_observed"] >= MIN_OBS) &
    (gap_ari["max_gap_length"] <= MAX_GAP)
]["location"].tolist()

ok_ili = gap_ili[
    (gap_ili["n_observed"] >= MIN_OBS) &
    (gap_ili["max_gap_length"] <= MAX_GAP)
]["location"].tolist()

common_locations = sorted(set(ok_ari).intersection(set(ok_ili)))

print("ok_ari:", sorted(ok_ari))
print("ok_ili:", sorted(ok_ili))
print("common_locations:", common_locations)
print("n common:", len(common_locations))


# ============================================================
# 2. Calendar and preprocessing helpers
# ============================================================

full_weeks = pd.date_range(
    start=pd.Timestamp("2014-01-05"),
    end=max(ari_df["truth_date"].max(), ili_df["truth_date"].max()),
    freq="W-SUN"
)

train_weeks = full_weeks[
    (full_weeks >= pd.Timestamp("2014-01-05")) &
    (full_weeks <= pd.Timestamp("2022-12-25"))
]

test_weeks = full_weeks[
    (full_weeks >= pd.Timestamp("2023-01-01")) &
    (full_weeks <= pd.Timestamp("2023-12-31"))
]

print("train weeks:", len(train_weeks), train_weeks.min(), "->", train_weeks.max())
print("test weeks :", len(test_weeks), test_weeks.min(), "->", test_weeks.max())


def build_bivariate_train_and_test(location: str):
    """
    Build:
      - imputed bivariate training panel [ARI, ILI]
      - raw bivariate 2023 test truth [ARI, ILI]
      - disease-specific MASE scales computed on training
    """

    ari_train = impute_series_weekly(
        ari_df[ari_df["location"] == location][["truth_date", "value"]],
        calendar=train_weeks,
        trim_to_first_obs=True,
        max_gap_knn=8
    )

    ili_train = impute_series_weekly(
        ili_df[ili_df["location"] == location][["truth_date", "value"]],
        calendar=train_weeks,
        trim_to_first_obs=True,
        max_gap_knn=8
    )

    train_bi = pd.concat(
        [
            ari_train.rename("ARI"),
            ili_train.rename("ILI")
        ],
        axis=1
    ).dropna()

    ari_test = extract_raw_weekly_series(ari_df, location, test_weeks).rename("ARI")
    ili_test = extract_raw_weekly_series(ili_df, location, test_weeks).rename("ILI")

    test_bi = pd.concat([ari_test, ili_test], axis=1)

    scale_ari = mase_scale_seasonal(train_bi["ARI"], m=M_SEASON)
    scale_ili = mase_scale_seasonal(train_bi["ILI"], m=M_SEASON)

    return train_bi, test_bi, scale_ari, scale_ili


def make_diff_row(new_row: pd.DataFrame, prev_level_row: pd.Series) -> pd.DataFrame:
    """
    For d=1 models:
    convert a new observed level row into a differenced row.
    Missing values remain missing.
    """

    out = new_row.copy()

    for col in out.columns:
        new_val = out.iloc[0][col]
        prev_val = prev_level_row[col]

        if pd.notna(new_val) and pd.notna(prev_val):
            out.iloc[0, out.columns.get_loc(col)] = float(new_val - prev_val)
        else:
            out.iloc[0, out.columns.get_loc(col)] = np.nan

    return out


# ============================================================
# 3. Model fitting and specification search
# ============================================================

def fit_varima_candidate(train_bi: pd.DataFrame, p: int, d: int, q: int, trend="c", maxiter=500):
    """
    Fit a bivariate VARIMA-like model using VARMAX.

    d=0: VARMAX on levels
    d=1: VARMAX on first differences

    Returns fitted result, AIC and convergence flag.
    """

    if d == 0:
        endog = train_bi.copy()
    elif d == 1:
        endog = train_bi.diff().dropna().copy()
    else:
        raise ValueError("Only d in {0,1} is supported.")

    if len(endog) < 80:
        raise ValueError("Too few observations after preprocessing.")

    model = VARMAX(
        endog,
        order=(p, q),
        trend=trend,
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    res = model.fit(disp=False, maxiter=maxiter)

    converged = bool(res.mle_retvals.get("converged", False))
    aic = float(res.aic) if np.isfinite(res.aic) else np.nan

    return res, aic, converged


CANDIDATES = [
    {"p": 1, "d": 0, "q": 0},
    {"p": 2, "d": 0, "q": 0},
    {"p": 1, "d": 0, "q": 1},
    {"p": 2, "d": 0, "q": 1},
    {"p": 1, "d": 1, "q": 0},
    {"p": 2, "d": 1, "q": 0},
    {"p": 1, "d": 1, "q": 1},
    {"p": 2, "d": 1, "q": 1},
]

MODEL_NAME = "bivariate_varima_rolling_2023"


def select_best_bivariate_spec(train_bi: pd.DataFrame, location: str):
    """
    Search over a small set of VARIMA-like specifications.

    Selection rule:
      1. Prefer the lowest-AIC model among converged candidates.
      2. If no candidate converged, use the lowest finite AIC as fallback.
         This is recorded in the search table.
    """

    rows = []
    fitted_objects = {}

    for cand in CANDIDATES:
        p, d, q = cand["p"], cand["d"], cand["q"]
        key = (p, d, q)

        try:
            res, aic, converged = fit_varima_candidate(
                train_bi=train_bi,
                p=p,
                d=d,
                q=q,
                trend="c",
                maxiter=500
            )

            fitted_objects[key] = res

            rows.append({
                "location": location,
                "p": p,
                "d": d,
                "q": q,
                "aic": aic,
                "converged": int(converged),
                "ok": 1,
                "error": ""
            })

        except Exception as e:
            rows.append({
                "location": location,
                "p": p,
                "d": d,
                "q": q,
                "aic": np.nan,
                "converged": 0,
                "ok": 0,
                "error": str(e)[:300]
            })

    search_df = pd.DataFrame(rows)

    valid_conv = search_df[
        (search_df["ok"] == 1) &
        (search_df["converged"] == 1) &
        (search_df["aic"].notna())
    ].copy()

    valid_any = search_df[
        (search_df["ok"] == 1) &
        (search_df["aic"].notna())
    ].copy()

    if len(valid_conv) > 0:
        selected = valid_conv.sort_values("aic").iloc[0]
        selection_status = "selected_converged"
    elif len(valid_any) > 0:
        selected = valid_any.sort_values("aic").iloc[0]
        selection_status = "selected_fallback_nonconverged"
    else:
        search_df["selected"] = 0
        search_df["selection_status"] = "no_valid_model"
        return search_df, None, None

    best_spec = {
        "p": int(selected["p"]),
        "d": int(selected["d"]),
        "q": int(selected["q"])
    }

    best_key = (best_spec["p"], best_spec["d"], best_spec["q"])
    best_res = fitted_objects[best_key]

    search_df["selected"] = (
        (search_df["p"] == best_spec["p"]) &
        (search_df["d"] == best_spec["d"]) &
        (search_df["q"] == best_spec["q"])
    ).astype(int)

    search_df["selection_status"] = ""
    search_df.loc[search_df["selected"] == 1, "selection_status"] = selection_status

    return search_df, best_spec, best_res


# ============================================================
# 4. Correct rolling-origin evaluation
# ============================================================

def rolling_bivariate_varima_eval(
    train_bi: pd.DataFrame,
    test_bi: pd.DataFrame,
    spec: dict,
    fitted_res
):
    """
    Correct rolling-origin evaluation.

    Important details:
      - At each 2023 origin, the current observed week is appended to the
        state-space filter before forecasting future horizons.
      - Forecasts are generated for horizons 1..4.
      - Rows are stored for every target date inside the 2023 test calendar.
      - Missing actuals or missing predictions are NOT filtered here.
        They are filtered only when computing metrics.
      - This makes n_points auditable and consistent.
    """

    p, d, q = spec["p"], spec["d"], spec["q"]

    res = fitted_res
    rows = []

    last_level = train_bi.iloc[-1].copy()

    for origin in test_bi.index:

        new_obs = test_bi.loc[[origin], ["ARI", "ILI"]].copy()

        # ----------------------------------------------------
        # 1. Update state with the current observed week
        # ----------------------------------------------------
        if d == 0:

            try:
                res = res.append(new_obs, refit=False)
            except Exception:
                # If append fails for any reason, keep the previous state.

                pass

            for col in ["ARI", "ILI"]:
                if pd.notna(new_obs.iloc[0][col]):
                    last_level[col] = float(new_obs.iloc[0][col])

            try:
                fc = res.forecast(steps=MAX_H).copy()
            except Exception:
                fc = pd.DataFrame(
                    np.nan,
                    index=range(MAX_H),
                    columns=["ARI", "ILI"]
                )

        elif d == 1:

            diff_obs = make_diff_row(new_obs, last_level)

            try:
                res = res.append(diff_obs, refit=False)
            except Exception:
                pass

            for col in ["ARI", "ILI"]:
                if pd.notna(new_obs.iloc[0][col]):
                    last_level[col] = float(new_obs.iloc[0][col])

            try:
                fc_diff = res.forecast(steps=MAX_H).copy()
                fc = fc_diff.cumsum().add(last_level, axis=1)
            except Exception:
                fc = pd.DataFrame(
                    np.nan,
                    index=range(MAX_H),
                    columns=["ARI", "ILI"]
                )

        else:
            raise ValueError("Only d in {0,1} is supported.")

        # ----------------------------------------------------
        # 2. Store all horizon-level rows inside test calendar
        # ----------------------------------------------------
        for h in HORIZONS:

            target = origin + pd.Timedelta(weeks=h)

            if target not in test_bi.index:
                continue

            for disease in ["ARI", "ILI"]:

                actual = test_bi.loc[target, disease]

                try:
                    pred = fc.iloc[h - 1][disease]
                except Exception:
                    pred = np.nan

                rows.append({
                    "origin": origin,
                    "target": target,
                    "location": None,  # filled after function call
                    "disease": disease,
                    "h": int(h),
                    "y": float(actual) if pd.notna(actual) else np.nan,
                    "pred": float(pred) if pd.notna(pred) else np.nan,
                    "p": int(p),
                    "d": int(d),
                    "q": int(q),
                    "model": MODEL_NAME
                })

    return pd.DataFrame(rows)


def summarize_country_h(
    df_long: pd.DataFrame,
    scale_ari: float,
    scale_ili: float,
    location: str
):
    """
    Compute country-horizon metrics.

    Filtering rule:
      - Metrics are computed only where y is observed and pred is finite.
      - n is the number of scored predictions.
    """

    out = []

    for disease, scale in [("ARI", scale_ari), ("ILI", scale_ili)]:

        sub_d = df_long[df_long["disease"] == disease].copy()

        for h in HORIZONS:

            sub = sub_d[sub_d["h"] == h].copy()

            # Correct scoring filter
            sub = sub[sub["y"].notna()].copy()
            sub = sub[sub["pred"].notna()].copy()

            if len(sub) == 0:
                continue

            m_mae = mae(sub["y"].values, sub["pred"].values)
            m_rmse = rmse(sub["y"].values, sub["pred"].values)
            m_mase = mase_from_mae(m_mae, scale)

            out.append({
                "disease": disease,
                "location": location,
                "h": int(h),
                "model": MODEL_NAME,
                "mae": m_mae,
                "rmse": m_rmse,
                "mase": m_mase,
                "n": int(len(sub))
            })

    return pd.DataFrame(out)


def expected_points_from_truth(test_bi: pd.DataFrame, location: str):
    """
    Independent audit of how many actual target values exist in the test set.

    This does not depend on the model. It only counts available ground truth
    values by disease and horizon.
    """

    rows = []

    for disease in ["ARI", "ILI"]:

        for h in HORIZONS:

            n_expected = 0

            for origin in test_bi.index:

                target = origin + pd.Timedelta(weeks=h)

                if target not in test_bi.index:
                    continue

                if pd.notna(test_bi.loc[target, disease]):
                    n_expected += 1

            rows.append({
                "disease": disease,
                "location": location,
                "h": int(h),
                "expected_actual_points": int(n_expected)
            })

    return pd.DataFrame(rows)


# ============================================================
# 5. Run benchmark
# ============================================================

all_search_rows = []
all_long_rows = []
all_country_h = []
all_expected_rows = []

for location in common_locations:

    print(f"\n==================== {location} ====================")

    train_bi, test_bi, scale_ari, scale_ili = build_bivariate_train_and_test(location)

    print("train_bi shape:", train_bi.shape)
    print("test_bi  shape:", test_bi.shape)

    expected_df = expected_points_from_truth(test_bi, location)
    all_expected_rows.append(expected_df)

    search_df, best_spec, best_res = select_best_bivariate_spec(train_bi, location)
    all_search_rows.append(search_df)

    if best_spec is None:
        print("No valid model found for", location)
        continue

    print("Best spec:", best_spec)

    df_long = rolling_bivariate_varima_eval(
        train_bi=train_bi,
        test_bi=test_bi,
        spec=best_spec,
        fitted_res=best_res
    )

    if len(df_long) == 0:
        print("No prediction rows generated for", location)
        continue

    df_long["location"] = location
    all_long_rows.append(df_long)

    df_country_h = summarize_country_h(
        df_long=df_long,
        scale_ari=scale_ari,
        scale_ili=scale_ili,
        location=location
    )

    all_country_h.append(df_country_h)

search_results = pd.concat(all_search_rows, ignore_index=True) if all_search_rows else pd.DataFrame()
varima_long = pd.concat(all_long_rows, ignore_index=True) if all_long_rows else pd.DataFrame()
varima_country_h = pd.concat(all_country_h, ignore_index=True) if all_country_h else pd.DataFrame()
expected_points = pd.concat(all_expected_rows, ignore_index=True) if all_expected_rows else pd.DataFrame()

print("\nDone.")
print("search_results:", search_results.shape)
print("varima_long:", varima_long.shape)
print("varima_country_h:", varima_country_h.shape)
print("expected_points:", expected_points.shape)


# ============================================================
# 6. Audit n_points
# ============================================================

scored_points = (
    varima_country_h
    .groupby(["disease", "location", "h"], as_index=False)
    .agg(scored_points=("n", "sum"))
)

n_audit = expected_points.merge(
    scored_points,
    on=["disease", "location", "h"],
    how="left"
)

n_audit["scored_points"] = n_audit["scored_points"].fillna(0).astype(int)
n_audit["missing_due_to_model_or_pred_nan"] = (
    n_audit["expected_actual_points"] - n_audit["scored_points"]
)

print("\nN audit by disease/location/h:")
display(n_audit)

print("\nN audit aggregated by disease/h:")
display(
    n_audit
    .groupby(["disease", "h"], as_index=False)
    .agg(
        expected_actual_points=("expected_actual_points", "sum"),
        scored_points=("scored_points", "sum"),
        missing_due_to_model_or_pred_nan=("missing_due_to_model_or_pred_nan", "sum")
    )
)


# ============================================================
# 7. Build summary tables
# ============================================================

def build_summary_tables(country_h_df: pd.DataFrame, disease: str):

    sub = country_h_df[country_h_df["disease"] == disease].copy()

    country_summary = (
        sub.groupby(["location", "model"], as_index=False)
        .agg(
            mae=("mae", "mean"),
            rmse=("rmse", "mean"),
            mase=("mase", "mean"),
            n_points=("n", "sum")
        )
        .sort_values(["mase", "mae"])
        .reset_index(drop=True)
    )

    horizon_summary = (
        sub.groupby(["h", "model"], as_index=False)
        .agg(
            mae=("mae", "mean"),
            rmse=("rmse", "mean"),
            mase=("mase", "mean"),
            n_countries=("location", "nunique"),
            n_points=("n", "sum")
        )
        .sort_values(["h", "mase", "mae"])
        .reset_index(drop=True)
    )

    return country_summary, horizon_summary


ari_country_summary_varima, ari_horizon_summary_varima = build_summary_tables(
    varima_country_h,
    "ARI"
)

ili_country_summary_varima, ili_horizon_summary_varima = build_summary_tables(
    varima_country_h,
    "ILI"
)

print("\nARI horizon summary")
display(ari_horizon_summary_varima)

print("\nILI horizon summary")
display(ili_horizon_summary_varima)

print("\nARI country summary")
display(ari_country_summary_varima)

print("\nILI country summary")
display(ili_country_summary_varima)


# ============================================================
# 8. Save outputs
# ============================================================

search_results.to_csv(paths.results / "bivariate_varima_spec_search_2023.csv", index=False)
varima_long.to_csv(paths.results / "bivariate_varima_rolling_2023_long.csv", index=False)
varima_country_h.to_csv(paths.results / "bivariate_varima_country_h_2023.csv", index=False)
expected_points.to_csv(paths.results / "bivariate_varima_expected_points_2023.csv", index=False)
n_audit.to_csv(paths.results / "bivariate_varima_n_audit_2023.csv", index=False)

ari_country_summary_varima.to_csv(paths.results / "bivariate_varima_ari_country_2023.csv", index=False)
ili_country_summary_varima.to_csv(paths.results / "bivariate_varima_ili_country_2023.csv", index=False)

ari_horizon_summary_varima.to_csv(paths.results / "bivariate_varima_ari_horizon_2023.csv", index=False)
ili_horizon_summary_varima.to_csv(paths.results / "bivariate_varima_ili_horizon_2023.csv", index=False)

print("Saved to:", paths.results)


# ============================================================
# 9. Optional comparison with previous global summaries
# ============================================================

def try_read(path):
    try:
        return pd.read_csv(path)
    except Exception:
        return None


ari_prev = try_read(paths.results / "ari_summary_horizon.csv")
ili_prev = try_read(paths.results / "ili_summary_horizon.csv")

if ari_prev is not None:
    ari_prev = ari_prev.rename(columns={"MAE": "mae", "RMSE": "rmse", "MASE": "mase"})
    ari_compare = pd.concat(
        [ari_prev, ari_horizon_summary_varima],
        ignore_index=True
    ).sort_values(["h", "mase", "mae"]).reset_index(drop=True)

    print("\nARI comparison including bivariate VARIMA")
    display(ari_compare)

if ili_prev is not None:
    ili_prev = ili_prev.rename(columns={"MAE": "mae", "RMSE": "rmse", "MASE": "mase"})
    ili_compare = pd.concat(
        [ili_prev, ili_horizon_summary_varima],
        ignore_index=True
    ).sort_values(["h", "mase", "mae"]).reset_index(drop=True)

    print("\nILI comparison including bivariate VARIMA")
    display(ili_compare)

ARI shape: (6685, 4)
ILI shape: (10646, 4)
ok_ari: ['BE', 'BG', 'CZ', 'DE', 'EE', 'LT', 'RO', 'SI']
ok_ili: ['BE', 'CZ', 'EE', 'GR', 'IE', 'LT', 'NL', 'PL', 'RO', 'SI']
common_locations: ['BE', 'CZ', 'EE', 'LT', 'RO', 'SI']
n common: 6
train weeks: 469 2014-01-05 00:00:00 -> 2022-12-25 00:00:00
test weeks : 53 2023-01-01 00:00:00 -> 2023-12-31 00:00:00

==================== BE ====================
train_bi shape: (430, 2)
test_bi  shape: (53, 2)


C:\Users\aolas\miniconda3\Lib\site-packages\statsmodels\tsa\statespace\varmax.py:160: EstimationWarning: Estimation of VARMA(p,q) models is not generically robust, due especially to identification issues.
  warn('Estimation of VARMA(p,q) models is not generically robust,'
C:\Users\aolas\miniconda3\Lib\site-packages\statsmodels\tsa\statespace\varmax.py:160: EstimationWarning: Estimation of VARMA(p,q) models is not generically robust, due especially to identification issues.
  warn('Estimation of VARMA(p,q) models is not generically robust,'
C:\Users\aolas\miniconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\aolas\miniconda3\Lib\site-packages\statsmodels\tsa\statespace\varmax.py:160: EstimationWarning: Estimation of VARMA(p,q) models is not generically robust, due especially to identification issues.
  warn('Estimation of V

Best spec: {'p': 1, 'd': 0, 'q': 1}


C:\Users\aolas\miniconda3\Lib\site-packages\statsmodels\tsa\statespace\varmax.py:160: EstimationWarning: Estimation of VARMA(p,q) models is not generically robust, due especially to identification issues.
  warn('Estimation of VARMA(p,q) models is not generically robust,'
C:\Users\aolas\miniconda3\Lib\site-packages\statsmodels\tsa\statespace\varmax.py:160: EstimationWarning: Estimation of VARMA(p,q) models is not generically robust, due especially to identification issues.
  warn('Estimation of VARMA(p,q) models is not generically robust,'
C:\Users\aolas\miniconda3\Lib\site-packages\statsmodels\tsa\statespace\varmax.py:160: EstimationWarning: Estimation of VARMA(p,q) models is not generically robust, due especially to identification issues.
  warn('Estimation of VARMA(p,q) models is not generically robust,'
C:\Users\aolas\miniconda3\Lib\site-packages\statsmodels\tsa\statespace\varmax.py:160: EstimationWarning: Estimation of VARMA(p,q) models is not generically robust, due especially to


==================== CZ ====================
train_bi shape: (430, 2)
test_bi  shape: (53, 2)


...


ARI country summary


,location,model,mae,rmse,mase,n_points
0,BE,bivariate_varima_rolling_2023,200.348695,242.879327,0.675023,202
1,CZ,bivariate_varima_rolling_2023,129.604985,182.148463,0.678695,202
2,SI,bivariate_varima_rolling_2023,288.602575,382.549949,0.752689,202
3,EE,bivariate_varima_rolling_2023,60.557191,87.934700,0.895692,198
4,RO,bivariate_varima_rolling_2023,134.152235,167.552585,0.980229,186
5,LT,bivariate_varima_rolling_2023,179.196932,252.798777,1.148730,198



ILI country summary


,location,model,mae,rmse,mase,n_points
0,CZ,bivariate_varima_rolling_2023,9.221118,18.908648,0.460724,202
1,BE,bivariate_varima_rolling_2023,71.117700,100.824314,0.669120,202
2,RO,bivariate_varima_rolling_2023,2.130389,3.301537,0.817094,186
3,SI,bivariate_varima_rolling_2023,12.498960,26.416660,0.963851,202
4,LT,bivariate_varima_rolling_2023,24.212982,30.554359,1.343073,198
5,EE,bivariate_varima_rolling_2023,56.277431,82.137577,1.586912,198


Saved to: C:\Users\aolas\UNI PYTHON\TFG\results
